In [1]:
import zarr
import re
import os
import gc
import time

import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import dask.array as da

from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar
from sklearn.model_selection import train_test_split

In [2]:
def sanitize_key(key):
    return re.sub(r'[^\w]', '_', str(key)).strip('_')

**Choose HVGs**

In [3]:
adata = sc.read_h5ad('/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/kang_2018.h5ad')

In [4]:
rng = np.random.default_rng(seed=42)

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=5000)

all_data_hvg_names = adata.var[adata.var.highly_variable].index.tolist()
idx = np.sort(rng.choice(np.arange(adata.shape[1]), size=5000))
random_hvg_names = adata.var.iloc[idx].index.tolist()

In [5]:
adata.obs['cell_id'] = adata.obs.index
adata.obs = adata.obs.reset_index(drop=True)

adata.obs.loc[:, 'combination_key'] = adata.obs.apply(
    lambda row: f"{sanitize_key(row['cell_type'])}-"
                f"{sanitize_key(row['replicate'])}-"
                f"{sanitize_key(row['label'])}",
    axis=1
)

In [6]:
train_hvgs = dict()
for split in ['split01', 'split02', 'split03']:
    train_combinations = zarr.open_group(f'/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/{split}/split_metadata.zarr', mode='r').attrs["train_combinations"]
    
    adata.obs["is_train"] = adata.obs.apply(lambda x: x.combination_key in train_combinations, axis=1)
    tmp = adata[adata.obs.is_train].copy()
    sc.pp.highly_variable_genes(tmp, n_top_genes=5000)
    
    train_hvgs[split] = tmp.var[tmp.var.highly_variable].index.to_list()

/home/icb/egor.antipov/miniconda3/envs/scaling-transformers/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/icb/egor.antipov/miniconda3/envs/scaling-transformers/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/icb/egor.antipov/miniconda3/envs/scaling-transformers/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


**Prepare combinations**

In [12]:
output_dir = "/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang_w_ag/comb_w_obs.zarr"

In [13]:
os.makedirs(output_dir, exist_ok=True)
n_vars = adata.X.shape[1]
root_zarr = zarr.open(output_dir, mode='a')

OSError: [Errno 122] Disk quota exceeded

In [10]:
CHUNK_SIZE = 5_000
current_combinations = set()

for combination_key, group in tqdm(adata.obs.groupby('combination_key')):
    current_combinations.add(combination_key)
    x_group = adata.X[group.index].toarray()

    da.from_array(x_group, chunks=(CHUNK_SIZE, n_vars)).astype(adata.X.dtype).to_zarr(output_dir+'/'+combination_key, component='X', overwrite=True)
    da.from_array(group['cell_id'].values, chunks=-1).astype('<U40').to_zarr(output_dir+'/'+combination_key, component='obs_index', overwrite=True)

existing_combinations = set(root_zarr.attrs.get('combinations', []))
existing_combinations.update(current_combinations)
root_zarr.attrs['combinations'] = sorted(existing_combinations)

  0%|          | 0/124 [00:00<?, ?it/s]

**Best seed choice for split01**

In [11]:
train_ratio=0.7
val_ratio=0.15
test_ratio=0.15

for seed in range(30, 51):
    print("SEED:", seed)
    print(f"Processing combinations from {output_dir}")

    comb_dir = output_dir
    if not os.path.isdir(comb_dir):
        raise ValueError(f"Combination directory not found: {comb_dir}")

    combinations = [d for d in os.listdir(comb_dir) if not d.startswith('.')]
    cell_types = np.unique([d.split("-")[0] for d in combinations])
    print(f"Found {len(combinations)} combination directories")
    print(f"Found {len(cell_types)} cell types")

    train_val_ratio = train_ratio + val_ratio
    train_val_types, test_types = train_test_split(
        cell_types, 
        train_size=train_val_ratio,
        random_state=seed
    )

    train_types, val_types = train_test_split(
        train_val_types,
        train_size=train_ratio/train_val_ratio,
        random_state=seed
    )

    train_combs = []
    test_combs = []
    val_combs = []
    for combination in combinations:
        if combination.split("-")[0] in train_types:
            train_combs.append(combination)
        elif combination.split("-")[0] in test_types:
            test_combs.append(combination)
        elif combination.split("-")[0] in val_types:
            val_combs.append(combination)
        else:
            raise Exception(f"cell_type {combination.split('-')[0]} not found")

    print(f"Split: {len(train_combs)} train, {len(val_combs)} val, {len(test_combs)} test")

    val_arrays = []
    for comb in val_combs:
        comb_path = os.path.join(comb_dir, comb)
        val_arrays.append(da.from_zarr(comb_path, 'X'))

    val_data = da.concatenate(val_arrays, axis=0)
    print(f"Validation data shape: {val_data.shape}")

    test_arrays = []
    for comb in test_combs:
        comb_path = os.path.join(comb_dir, comb)
        test_arrays.append(da.from_zarr(comb_path, 'X'))

    test_data = da.concatenate(test_arrays,  axis=0)
    print(f"Test data shape: {test_data.shape}")
    print(f"Train data shape: ({len(adata) - test_data.shape[0] - val_data.shape[0]}, 5412)")
    print()

SEED: 30
Processing combinations from /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/comb_w_obs.zarr
Found 124 combination directories
Found 8 cell types
Split: 64 train, 28 val, 32 test


/home/icb/egor.antipov/miniconda3/envs/scaling-transformers/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


Validation data shape: (2783, 5000)
Test data shape: (7318, 5000)
Train data shape: (14572, 5412)

SEED: 31
Processing combinations from /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/comb_w_obs.zarr
Found 124 combination directories
Found 8 cell types
Split: 64 train, 28 val, 32 test
Validation data shape: (2783, 5000)
Test data shape: (6226, 5000)
Train data shape: (15664, 5412)

SEED: 32
Processing combinations from /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/comb_w_obs.zarr
Found 124 combination directories
Found 8 cell types
Split: 64 train, 32 val, 28 test
Validation data shape: (12954, 5000)
Test data shape: (5829, 5000)
Train data shape: (5890, 5412)

SEED: 33
Processing combinations from /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/comb_w_obs.zarr
Found 124 combination directories
Found 8 cell types
Split: 64 train, 32 val, 28 test
Validation data shape: (6226, 5000)
Test data shape: (1848, 5000)
Train data shape

In [12]:
# 36, 39

**Prepare split01**

In [13]:
CHUNK_SIZE = 5_000

source_dir = '/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang_w_ag/comb_w_obs.zarr'
output_dir = "/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang_w_ag/split01"

train_ratio=0.7
val_ratio=0.15
test_ratio=0.15

seed=39

print(f"Processing combinations from {source_dir}")
os.makedirs(output_dir, exist_ok=True)

comb_dir = source_dir
if not os.path.isdir(comb_dir):
    raise ValueError(f"Combination directory not found: {comb_dir}")

combinations = [d for d in os.listdir(comb_dir) if not d.startswith('.')]
cell_types = np.unique([d.split("-")[0] for d in combinations])
print(f"Found {len(combinations)} combination directories")
print(f"Found {len(cell_types)} cell types")

train_val_ratio = train_ratio + val_ratio
train_val_types, test_types = train_test_split(
    cell_types, 
    train_size=train_val_ratio,
    random_state=seed
)

train_types, val_types = train_test_split(
    train_val_types,
    train_size=train_ratio/train_val_ratio,
    random_state=seed
)

train_combs = []
test_combs = []
val_combs = []
for combination in combinations:
    if combination.split("-")[0] in train_types:
        train_combs.append(combination)
    elif combination.split("-")[0] in test_types:
        test_combs.append(combination)
    elif combination.split("-")[0] in val_types:
        val_combs.append(combination)
    else:
        raise Exception(f"cell_type {combination.split('-')[0]} not found")

print(f"Split: {len(train_combs)} train, {len(val_combs)} val, {len(test_combs)} test")

meta_save_path = os.path.join(output_dir, "split_metadata.zarr")
meta_group = zarr.open_group(meta_save_path, mode='w')
meta_group.attrs['train_combinations'] = train_combs
meta_group.attrs['val_combinations'] = val_combs
meta_group.attrs['test_combinations'] = test_combs
meta_group.attrs['random_seed'] = seed
meta_group.attrs['all_data_hvg_names'] = all_data_hvg_names
meta_group.attrs['random_hvg_names'] = random_hvg_names
meta_group.attrs['train_data_hvg_names'] = train_hvgs['split01']

print("Processing validation data...")
val_arrays = []
for comb in val_combs:
    comb_path = os.path.join(comb_dir, comb)
    val_arrays.append(da.from_zarr(comb_path, 'X'))

val_data = da.concatenate(val_arrays, axis=0)
print(f"Validation data shape: {val_data.shape}")

val_save_path = os.path.join(output_dir, 'val.zarr')
print(f"Saving validation data to {val_save_path}")
with ProgressBar():
    val_data.rechunk((CHUNK_SIZE, val_data.shape[1])).to_zarr(val_save_path, component='X', overwrite=True)
print(f"Done saving validation data")

del val_arrays, val_data
gc.collect()
time.sleep(10)

print("Processing test data...")
test_arrays = []
for comb in test_combs:
    comb_path = os.path.join(comb_dir, comb)
    test_arrays.append(da.from_zarr(comb_path, 'X'))

test_data = da.concatenate(test_arrays,  axis=0)
print(f"Test data shape: {test_data.shape}")

test_save_path = os.path.join(output_dir, 'test.zarr')
print(f"Saving test data to {test_save_path}")
with ProgressBar():
    test_data.rechunk((CHUNK_SIZE, test_data.shape[1])).to_zarr(test_save_path, component='X', overwrite=True)
print(f"Done saving test data")

del test_arrays, test_data
gc.collect()
time.sleep(10)

print("Processing training data...")
train_arrays = []
for comb in train_combs:
    comb_path = os.path.join(comb_dir, comb)
    train_arrays.append(da.from_zarr(comb_path, 'X'))

train_data = da.concatenate(train_arrays, axis=0)
print(f"Training data shape: {train_data.shape}")

n_vars = train_data.shape[1]
train_data = train_data.rechunk((CHUNK_SIZE, n_vars))
print(f"Training data rechunked: {train_data.numblocks} blocks")

train_unsfd_save_path = os.path.join(output_dir, 'train_unsfd.zarr')
print(f"Saving unshuffled training data to {train_unsfd_save_path}")
with ProgressBar():
    train_data.to_zarr(train_unsfd_save_path, component='X', overwrite=True)
print(f"Done saving unshuffled training data")

print("Shuffling training data...")
rng = np.random.default_rng(seed=seed)
n = train_data.shape[0]
perm = rng.permutation(n)
indexer = [list(perm[i:i+CHUNK_SIZE]) for i in range(0, n, CHUNK_SIZE)]

train_data_shuffled = da.shuffle(train_data, indexer=indexer, axis=0, chunks='auto')
print(f"Training data shuffled: {train_data_shuffled.numblocks} blocks")

train_save_path = os.path.join(output_dir, 'train.zarr')
with ProgressBar():
    train_data_shuffled.to_zarr(train_save_path, component='X', overwrite=True)
print(f"Done saving shuffled training data")

del train_arrays, train_data, train_data_shuffled
gc.collect()
time.sleep(10)

Processing combinations from /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/comb_w_obs.zarr
Found 124 combination directories
Found 8 cell types
Split: 64 train, 28 val, 32 test
Processing validation data...
Validation data shape: (2783, 5000)
Saving validation data to /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/split01/val.zarr
[########################################] | 100% Completed | 203.93 ms
Done saving validation data
Processing test data...
Test data shape: (2710, 5000)
Saving test data to /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/split01/test.zarr
[########################################] | 100% Completed | 204.50 ms
Done saving test data
Processing training data...
Training data shape: (19180, 5000)
Training data rechunked: (4, 1) blocks
Saving unshuffled training data to /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/split01/train_unsfd.zarr
[########################################] |

**Best seed choice for split02**

In [14]:
for seed in range(40, 51):
    print("SEED:", seed)
    print(f"Processing combinations from {source_dir}")

    comb_dir = source_dir
    if not os.path.isdir(comb_dir):
        raise ValueError(f"Combination directory not found: {comb_dir}")

    combinations = [d for d in os.listdir(comb_dir) if not d.startswith('.')]
    donors = np.unique([d.split("-")[1] for d in combinations])
    print(f"Found {len(combinations)} combination directories")
    print(f"Found {len(donors)} donors")

    train_val_ratio = train_ratio + val_ratio
    train_val_donors, test_donors = train_test_split(
        donors, 
        train_size=train_val_ratio,
        random_state=seed
    )

    train_donors, val_donors = train_test_split(
        train_val_donors,
        train_size=train_ratio/train_val_ratio,
        random_state=seed
    )

    train_combs = []
    test_combs = []
    val_combs = []
    for combination in combinations:
        if combination.split("-")[1] in train_donors:
            train_combs.append(combination)
        elif combination.split("-")[1] in test_donors:
            test_combs.append(combination)
        elif combination.split("-")[1] in val_donors:
            val_combs.append(combination)
        else:
            raise Exception(f"cell_type {combination.split('-')[1]} not found")

    print(f"Split: {len(train_combs)} train, {len(val_combs)} val, {len(test_combs)} test")

    val_arrays = []
    for comb in val_combs:
        comb_path = os.path.join(comb_dir, comb)
        val_arrays.append(da.from_zarr(comb_path, 'X'))

    val_data = da.concatenate(val_arrays, axis=0)
    print(f"Validation data shape: {val_data.shape}")

    test_arrays = []
    for comb in test_combs:
        comb_path = os.path.join(comb_dir, comb)
        test_arrays.append(da.from_zarr(comb_path, 'X'))

    test_data = da.concatenate(test_arrays,  axis=0)
    print(f"Test data shape: {test_data.shape}")
    print(f"Train data shape: ({len(adata) - test_data.shape[0] - val_data.shape[0]}, 5412)")
    print()

SEED: 40
Processing combinations from /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/comb_w_obs.zarr
Found 124 combination directories
Found 8 donors
Split: 62 train, 31 val, 31 test
Validation data shape: (5176, 5000)
Test data shape: (9670, 5000)
Train data shape: (9827, 5412)

SEED: 41
Processing combinations from /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/comb_w_obs.zarr
Found 124 combination directories
Found 8 donors
Split: 61 train, 31 val, 32 test
Validation data shape: (5682, 5000)
Test data shape: (7477, 5000)
Train data shape: (11514, 5412)

SEED: 42
Processing combinations from /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/comb_w_obs.zarr
Found 124 combination directories
Found 8 donors
Split: 62 train, 30 val, 32 test
Validation data shape: (6604, 5000)
Test data shape: (8433, 5000)
Train data shape: (9636, 5412)

SEED: 43
Processing combinations from /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruct

In [15]:
# 43 

**Prepare split02**

In [16]:
CHUNK_SIZE = 5_000

source_dir = '/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang_w_ag/comb_w_obs.zarr'
output_dir = "/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang_w_ag/split02"

train_ratio=0.7
val_ratio=0.15
test_ratio=0.15

seed=43

print(f"Processing combinations from {source_dir}")
os.makedirs(output_dir, exist_ok=True)

comb_dir = source_dir
if not os.path.isdir(comb_dir):
    raise ValueError(f"Combination directory not found: {comb_dir}")

combinations = [d for d in os.listdir(comb_dir) if not d.startswith('.')]
donors = np.unique([d.split("-")[1] for d in combinations])
print(f"Found {len(combinations)} combination directories")
print(f"Found {len(donors)} donors")

train_val_ratio = train_ratio + val_ratio
train_val_donors, test_donors = train_test_split(
    donors, 
    train_size=train_val_ratio,
    random_state=seed
)

train_donors, val_donors = train_test_split(
    train_val_donors,
    train_size=train_ratio/train_val_ratio,
    random_state=seed
)

train_combs = []
test_combs = []
val_combs = []
for combination in combinations:
    if combination.split("-")[1] in train_donors:
        train_combs.append(combination)
    elif combination.split("-")[1] in test_donors:
        test_combs.append(combination)
    elif combination.split("-")[1] in val_donors:
        val_combs.append(combination)
    else:
        raise Exception(f"cell_type {combination.split('-')[1]} not found")

print(f"Split: {len(train_combs)} train, {len(val_combs)} val, {len(test_combs)} test")

meta_save_path = os.path.join(output_dir, "split_metadata.zarr")
meta_group = zarr.open_group(meta_save_path, mode='w')
meta_group.attrs['train_combinations'] = train_combs
meta_group.attrs['val_combinations'] = val_combs
meta_group.attrs['test_combinations'] = test_combs
meta_group.attrs['random_seed'] = seed
meta_group.attrs['all_data_hvg_names'] = all_data_hvg_names
meta_group.attrs['random_hvg_names'] = random_hvg_names
meta_group.attrs['train_data_hvg_names'] = train_hvgs['split02']

print("Processing validation data...")
val_arrays = []
for comb in val_combs:
    comb_path = os.path.join(comb_dir, comb)
    val_arrays.append(da.from_zarr(comb_path, 'X'))

val_data = da.concatenate(val_arrays, axis=0)
print(f"Validation data shape: {val_data.shape}")

val_save_path = os.path.join(output_dir, 'val.zarr')
print(f"Saving validation data to {val_save_path}")
with ProgressBar():
    val_data.rechunk((CHUNK_SIZE, val_data.shape[1])).to_zarr(val_save_path, component='X', overwrite=True)
print(f"Done saving validation data")

del val_arrays, val_data
gc.collect()
time.sleep(10)

print("Processing test data...")
test_arrays = []
for comb in test_combs:
    comb_path = os.path.join(comb_dir, comb)
    test_arrays.append(da.from_zarr(comb_path, 'X'))

test_data = da.concatenate(test_arrays,  axis=0)
print(f"Test data shape: {test_data.shape}")

test_save_path = os.path.join(output_dir, 'test.zarr')
print(f"Saving test data to {test_save_path}")
with ProgressBar():
    test_data.rechunk((CHUNK_SIZE, test_data.shape[1])).to_zarr(test_save_path, component='X', overwrite=True)
print(f"Done saving test data")

del test_arrays, test_data
gc.collect()
time.sleep(10)

print("Processing training data...")
train_arrays = []
for comb in train_combs:
    comb_path = os.path.join(comb_dir, comb)
    train_arrays.append(da.from_zarr(comb_path, 'X'))

train_data = da.concatenate(train_arrays, axis=0)
print(f"Training data shape: {train_data.shape}")

n_vars = train_data.shape[1]
train_data = train_data.rechunk((CHUNK_SIZE, n_vars))
print(f"Training data rechunked: {train_data.numblocks} blocks")

train_unsfd_save_path = os.path.join(output_dir, 'train_unsfd.zarr')
print(f"Saving unshuffled training data to {train_unsfd_save_path}")
with ProgressBar():
    train_data.to_zarr(train_unsfd_save_path, component='X', overwrite=True)
print(f"Done saving unshuffled training data")

print("Shuffling training data...")
rng = np.random.default_rng(seed=seed)
n = train_data.shape[0]
perm = rng.permutation(n)
indexer = [list(perm[i:i+CHUNK_SIZE]) for i in range(0, n, CHUNK_SIZE)]

train_data_shuffled = da.shuffle(train_data, indexer=indexer, axis=0, chunks='auto')
print(f"Training data shuffled: {train_data_shuffled.numblocks} blocks")

train_save_path = os.path.join(output_dir, 'train.zarr')
with ProgressBar():
    train_data_shuffled.to_zarr(train_save_path, component='X', overwrite=True)
print(f"Done saving shuffled training data")

del train_arrays, train_data, train_data_shuffled
gc.collect()
time.sleep(10)

Processing combinations from /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/comb_w_obs.zarr
Found 124 combination directories
Found 8 donors
Split: 63 train, 30 val, 31 test
Processing validation data...
Validation data shape: (4400, 5000)
Saving validation data to /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/split02/val.zarr
[########################################] | 100% Completed | 204.44 ms
Done saving validation data
Processing test data...
Test data shape: (5682, 5000)
Saving test data to /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/split02/test.zarr
[########################################] | 100% Completed | 205.57 ms
Done saving test data
Processing training data...
Training data shape: (14591, 5000)
Training data rechunked: (3, 1) blocks
Saving unshuffled training data to /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/split02/train_unsfd.zarr
[########################################] | 100

**Prepare split03**

In [17]:
CHUNK_SIZE = 5_000

source_dir = '/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang_w_ag/comb_w_obs.zarr'
output_dir = "/lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang_w_ag/split03"

train_ratio=0.7
val_ratio=0.15
test_ratio=0.15

seed=42

print(f"Processing combinations from {source_dir}")
os.makedirs(output_dir, exist_ok=True)

comb_dir = source_dir
if not os.path.isdir(comb_dir):
    raise ValueError(f"Combination directory not found: {comb_dir}")

combinations = [d for d in os.listdir(comb_dir) if not d.startswith('.')]
print(f"Found {len(combinations)} combination directories")

train_val_ratio = train_ratio + val_ratio
train_val_combs, test_combs = train_test_split(
    combinations, 
    train_size=train_val_ratio,
    random_state=seed
)

train_combs, val_combs = train_test_split(
    train_val_combs,
    train_size=train_ratio/train_val_ratio,
    random_state=seed
)

print(f"Split: {len(train_combs)} train, {len(val_combs)} val, {len(test_combs)} test")

meta_save_path = os.path.join(output_dir, "split_metadata.zarr")
meta_group = zarr.open_group(meta_save_path, mode='w')
meta_group.attrs['train_combinations'] = train_combs
meta_group.attrs['val_combinations'] = val_combs
meta_group.attrs['test_combinations'] = test_combs
meta_group.attrs['random_seed'] = seed
meta_group.attrs['all_data_hvg_names'] = all_data_hvg_names
meta_group.attrs['random_hvg_names'] = random_hvg_names
meta_group.attrs['train_data_hvg_names'] = train_hvgs['split03']

print("Processing validation data...")
val_arrays = []
for comb in val_combs:
    comb_path = os.path.join(comb_dir, comb)
    val_arrays.append(da.from_zarr(comb_path, 'X'))

val_data = da.concatenate(val_arrays, axis=0)
print(f"Validation data shape: {val_data.shape}")

val_save_path = os.path.join(output_dir, 'val.zarr')
print(f"Saving validation data to {val_save_path}")
with ProgressBar():
    val_data.rechunk((CHUNK_SIZE, val_data.shape[1])).to_zarr(val_save_path, component='X', overwrite=True)
print(f"Done saving validation data")

del val_arrays, val_data
gc.collect()
time.sleep(10)

print("Processing test data...")
test_arrays = []
for comb in test_combs:
    comb_path = os.path.join(comb_dir, comb)
    test_arrays.append(da.from_zarr(comb_path, 'X'))

test_data = da.concatenate(test_arrays,  axis=0)
print(f"Test data shape: {test_data.shape}")

test_save_path = os.path.join(output_dir, 'test.zarr')
print(f"Saving test data to {test_save_path}")
with ProgressBar():
    test_data.rechunk((CHUNK_SIZE, test_data.shape[1])).to_zarr(test_save_path, component='X', overwrite=True)
print(f"Done saving test data")

del test_arrays, test_data
gc.collect()
time.sleep(10)

print("Processing training data...")
train_arrays = []
for comb in train_combs:
    comb_path = os.path.join(comb_dir, comb)
    train_arrays.append(da.from_zarr(comb_path, 'X'))

train_data = da.concatenate(train_arrays, axis=0)
print(f"Training data shape: {train_data.shape}")

n_vars = train_data.shape[1]
train_data = train_data.rechunk((CHUNK_SIZE, n_vars))
print(f"Training data rechunked: {train_data.numblocks} blocks")

train_unsfd_save_path = os.path.join(output_dir, 'train_unsfd.zarr')
print(f"Saving unshuffled training data to {train_unsfd_save_path}")
with ProgressBar():
    train_data.to_zarr(train_unsfd_save_path, component='X', overwrite=True)
print(f"Done saving unshuffled training data")

print("Shuffling training data...")
rng = np.random.default_rng(seed=seed)
n = train_data.shape[0]
perm = rng.permutation(n)
indexer = [list(perm[i:i+CHUNK_SIZE]) for i in range(0, n, CHUNK_SIZE)]

train_data_shuffled = da.shuffle(train_data, indexer=indexer, axis=0, chunks='auto')
print(f"Training data shuffled: {train_data_shuffled.numblocks} blocks")

train_save_path = os.path.join(output_dir, 'train.zarr')
with ProgressBar():
    train_data_shuffled.to_zarr(train_save_path, component='X', overwrite=True)
print(f"Done saving shuffled training data")

del train_arrays, train_data, train_data_shuffled
gc.collect()
time.sleep(10)

Processing combinations from /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/comb_w_obs.zarr
Found 124 combination directories
Split: 86 train, 19 val, 19 test
Processing validation data...
Validation data shape: (2945, 5000)
Saving validation data to /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/split03/val.zarr
[########################################] | 100% Completed | 204.72 ms
Done saving validation data
Processing test data...
Test data shape: (5238, 5000)
Saving test data to /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/split03/test.zarr
[########################################] | 100% Completed | 205.79 ms
Done saving test data
Processing training data...
Training data shape: (16490, 5000)
Training data rechunked: (4, 1) blocks
Saving unshuffled training data to /lustre/groups/ml01/workspace/xiaotong.fu/data/reconstruction/kang/split03/train_unsfd.zarr
[########################################] | 100% Completed | 3